# RoadVision AI: End-to-End Perception Model & Safety Analytics

This notebook runs the complete RoadVision AI capstone baseline locally or on Google Colab:
1. **Dynamic Path Resolution**: Checks local project workspace, parent directories, or falls back to common directories to locate datasets.
2. **Upgraded CNN Model**: Implements a PyTorch-based vehicle classifier and regression model with convolutional blocks, batch normalization, dropout, and global pooling.
3. **Tesla safety EDA**: Filters and processes reported Tesla fatalities and Autopilot incidents, outputting clean distributions and summary analytics.


In [ ]:
import os
import zipfile
import json
import random
import shutil
import warnings
import time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
warnings.filterwarnings('ignore')

print("Imports and environment set up successfully.")


## 1. Local Path Configurations and Data Extraction

In [ ]:
# Base directory relative to where this notebook is executed
BASE = Path.cwd()
DATA = BASE / 'data'
IMG_DIR = DATA / 'Images'
OUT = BASE / 'outputs'
FIG = OUT / 'figures'
MODELS = BASE / 'models'

# Create directories
for p in [DATA, OUT, FIG, MODELS]:
    p.mkdir(parents=True, exist_ok=True)

def locate_data_file(filename, expected_rel_path):
    # 1. Check direct paths
    candidate = BASE / filename
    if candidate.exists():
        return candidate
    candidate = (BASE / expected_rel_path).resolve()
    if candidate.exists():
        return candidate
    
    # 2. Search parent directory/datasets structure (e.g. Downloads/Datasets)
    search_dirs = []
    parent_dir = BASE.parent
    if parent_dir.exists() and parent_dir != Path('/'):
        search_dirs.append(parent_dir / 'Datasets')
        search_dirs.append(parent_dir)
    elif BASE.name == 'content':  # Google Colab root workspace
        search_dirs.append(BASE)
        
    for search_root in search_dirs:
        if search_root.exists():
            try:
                for found_path in search_root.rglob(filename):
                    # Skip system/proc folders in Linux environments
                    if any(part in found_path.parts for part in ['proc', 'sys', 'dev']):
                        continue
                    if found_path.is_file():
                        print(f"Auto-located file '{filename}' at: {found_path}")
                        return found_path
            except (OSError, PermissionError):
                continue
    raise FileNotFoundError(f"Could not locate file '{filename}' automatically.")

# Locate data
try:
    zip_src = locate_data_file('Images.zip', '../Datasets/Capstone 1/Part 1/Images.zip')
    labels_src = locate_data_file('labels.csv', '../Datasets/Capstone 1/Part 1/labels.csv')
    deaths_src = locate_data_file('Tesla - Deaths.csv', '../Datasets/Capstone 1/Part 2/Tesla - Deaths.csv')
except FileNotFoundError as e:
    print(f"\n[WARNING] Dynamic search failed: {e}")
    print("If running in Colab, upload the files directly and set local paths below:")
    zip_src = BASE / 'Images.zip'
    labels_src = BASE / 'labels.csv'
    deaths_src = BASE / 'Tesla - Deaths.csv'

# Extract images
if zip_src.exists() and (not IMG_DIR.exists() or len(list(IMG_DIR.glob('*.jpg'))) < 100):
    print(f"Extracting images from {zip_src}...", flush=True)
    with zipfile.ZipFile(zip_src) as z:
        z.extractall(DATA)
    print("Extraction complete.", flush=True)
else:
    print("Images already extracted or Images.zip not found yet.")


## 2. Dataset Preprocessing and Data Splitting

In [ ]:
print('Preparing data...', flush=True)
labels = pd.read_csv(labels_src, header=None, names=['image_id', 'class', 'xmin', 'ymin', 'xmax', 'ymax'])
labels['image_id_str'] = labels.image_id.astype(int).map(lambda x: f'{x:08d}')

image_paths = sorted(IMG_DIR.glob('*.jpg'))
image_ids = {p.stem for p in image_paths}
labels = labels[labels.image_id_str.isin(image_ids)].copy()

# Drop duplicates (keep largest box per image)
labels['box_area'] = (labels.xmax - labels.xmin).clip(lower=0) * (labels.ymax - labels.ymin).clip(lower=0)
rep = labels.sort_values(['image_id_str', 'box_area'], ascending=[True, False]).drop_duplicates('image_id_str').copy()
rep['path'] = rep.image_id_str.map(lambda s: str(IMG_DIR / f'{s}.jpg'))
rep = rep[rep.path.map(lambda p: Path(p).exists())].reset_index(drop=True)

# Image dimensions
wh = []
for p in rep.path:
    with Image.open(p) as im:
        wh.append((im.width, im.height))
rep['width'] = [w for w, h in wh]
rep['height'] = [h for w, h in wh]

for src, dst, den in [('xmin', 'x1n', 'width'), ('ymin', 'y1n', 'height'), ('xmax', 'x2n', 'width'), ('ymax', 'y2n', 'height')]:
    rep[dst] = (rep[src] / rep[den]).clip(0, 1)
rep = rep[(rep.x2n > rep.x1n) & (rep.y2n > rep.y1n)].reset_index(drop=True)

# Subset selection
max_records = 300
rep_sub = rep.sample(min(max_records, len(rep)), random_state=SEED).copy()

cnt = rep_sub['class'].value_counts()
if (cnt >= 8).sum() < 2:
    top = rep['class'].value_counts().head(8).index
    rep_sub = rep[rep['class'].isin(top)].sample(min(max_records, len(rep[rep['class'].isin(top)])), random_state=SEED).copy()
else:
    rep_sub = rep_sub[rep_sub['class'].isin(cnt[cnt >= 8].index)].copy()

le = LabelEncoder()
rep_sub['class_id'] = le.fit_transform(rep_sub['class'])
classes = le.classes_.tolist()

# Splits
train_df, temp_df = train_test_split(rep_sub, test_size=0.30, random_state=SEED, stratify=rep_sub.class_id)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df.class_id)

for name, df in [('object_detection_training_table', rep), ('object_detection_modeling_subset', rep_sub), 
                 ('train_split', train_df), ('val_split', val_df), ('test_split', test_df)]:
    df.to_csv(OUT / f'{name}.csv', index=False)

summary = {
    'n_images_zip': len(image_paths),
    'n_label_rows_total': int(pd.read_csv(labels_src, header=None).shape[0]),
    'n_label_rows_matching_images': int(labels.shape[0]),
    'n_images_with_labels': int(rep.shape[0]),
    'n_modeling_records': int(rep_sub.shape[0]),
    'classes': classes
}
print(f"Data preprocessing complete. Target categories: {classes}")


### Visualize Class Distribution

In [ ]:
plt.figure(figsize=(10, 4))
labels['class'].value_counts().plot(kind='bar', color='teal', title='All Annotation Class Distribution')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG / 'annotation_class_distribution.png', dpi=150)
plt.show()

plt.figure(figsize=(10, 4))
rep_sub['class'].value_counts().plot(kind='bar', color='coral', title='Modeling Subset Class Distribution')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG / 'modeling_subset_class_distribution.png', dpi=150)
plt.show()


## 3. PyTorch Model Definition (Upgraded CNN)

In [ ]:
class DS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        im = Image.open(r.path).convert('RGB').resize((64, 64))
        arr = np.asarray(im, dtype=np.float32) / 255.0
        arr = (arr - 0.5) / 0.5
        return (torch.tensor(arr).permute(2, 0, 1).float(), 
                torch.tensor(int(r.class_id)).long(), 
                torch.tensor([r.x1n, r.y1n, r.x2n, r.y2n], dtype=torch.float32))

class UpgradedCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.f = nn.Sequential( 
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # -> 32x32
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # -> 16x16
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.AdaptiveAvgPool2d((2, 2)),  # -> 2x2
            nn.Flatten()  # -> 512 features
        )
        
        self.c = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
        
        self.b = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 4),
            nn.Sigmoid()
        )

    def forward(self, x):
        h = self.f(x)
        return self.c(h), self.b(h)

def iou(a, b):
    xA = torch.maximum(a[:, 0], b[:, 0])
    yA = torch.maximum(a[:, 1], b[:, 1])
    xB = torch.minimum(a[:, 2], b[:, 2])
    yB = torch.minimum(a[:, 3], b[:, 3])
    inter = (xB - xA).clamp(min=0) * (yB - yA).clamp(min=0)
    area1 = (a[:, 2] - a[:, 0]).clamp(min=0) * (a[:, 3] - a[:, 1]).clamp(min=0)
    area2 = (b[:, 2] - b[:, 0]).clamp(min=0) * (b[:, 3] - b[:, 1]).clamp(min=0)
    return inter / (area1 + area2 - inter + 1e-8)


## 4. Train and Evaluate the Upgraded Model

In [ ]:
train_loader = DataLoader(DS(train_df), batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(DS(val_df), batch_size=64)
test_loader = DataLoader(DS(test_df), batch_size=64)

model = UpgradedCNN(len(classes))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
ce = nn.CrossEntropyLoss()
sl = nn.SmoothL1Loss()

def eval_loader(loader):
    model.eval()
    tot = 0
    corr = 0
    losses = []
    ious = []
    with torch.no_grad():
        for x, y, b in loader:
            logits, pb = model(x)
            loss = ce(logits, y) + 5 * sl(pb, b)
            losses.append(loss.item() * len(x))
            tot += len(x)
            corr += (logits.argmax(1) == y).sum().item()
            ious += iou(pb, b).numpy().tolist()
    return {'loss': sum(losses)/tot, 'accuracy': corr/tot, 'mean_iou': float(np.mean(ious))}

print('Training model...', flush=True)
hist = []
epochs = 5
for ep in range(1, epochs + 1):
    model.train()
    total = 0
    n = 0
    for bi, (x, y, b) in enumerate(train_loader):
        opt.zero_grad()
        logits, pb = model(x)
        loss = ce(logits, y) + 5 * sl(pb, b)
        loss.backward()
        opt.step()
        total += loss.item() * len(x)
        n += len(x)
    vm = eval_loader(val_loader)
    rec = {'epoch': ep, 'train_loss': total/n, **{f'val_{k}': v for k, v in vm.items()}}
    hist.append(rec)
    print(rec, flush=True)

metrics = {'train': eval_loader(train_loader), 'validation': eval_loader(val_loader), 'test': eval_loader(test_loader), 'history': hist, 'summary': summary}
json.dump(metrics, open(OUT / 'model_metrics.json', 'w'), indent=2)
torch.save({'model_state_dict': model.state_dict(), 'classes': classes, 'input_size': 64, 'metrics': metrics}, MODELS / 'roadvision_tiny_detector.pt')


### Plot Training Performance Curves

In [ ]:
h = pd.DataFrame(hist)

plt.figure(figsize=(7, 4))
plt.plot(h.epoch, h.train_loss, marker='o', label='Train')
plt.plot(h.epoch, h.val_loss, marker='o', label='Validation')
plt.legend()
plt.title('Training Loss')
plt.savefig(FIG / 'training_loss_curve.png', dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(h.epoch, h.val_accuracy, marker='o', label='Validation Accuracy')
plt.plot(h.epoch, h.val_mean_iou, marker='o', label='Validation Mean IoU')
plt.legend()
plt.title('Validation Performance')
plt.savefig(FIG / 'validation_performance.png', dpi=150)
plt.show()


### Show Visual Predictions (Ground Truth vs Model Box)

In [ ]:
pred_dir = OUT / 'sample_predictions'
pred_dir.mkdir(exist_ok=True)
rows = []

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (_, r) in enumerate(test_df.sample(min(6, len(test_df)), random_state=SEED).iterrows()):
    x, _, _ = DS(pd.DataFrame([r]))[0]
    with torch.no_grad():
        logits, pb = model(x.unsqueeze(0))
        pr = torch.softmax(logits, 1).numpy()[0]
        pred_idx = int(pr.argmax())
        box = pb.numpy()[0]
    
    im = Image.open(r.path).convert('RGB')
    W, H = im.size
    dr = ImageDraw.Draw(im)
    true = [int(r.x1n * W), int(r.y1n * H), int(r.x2n * W), int(r.y2n * H)]
    pred = [int(box[0] * W), int(box[1] * H), int(box[2] * W), int(box[3] * H)]
    
    dr.rectangle(true, outline='green', width=3)
    dr.rectangle(pred, outline='red', width=3)
    
    title_text = f"true={r['class']}\npred={classes[pred_idx]} ({pr[pred_idx]:.2f})"
    axes[idx].imshow(im)
    axes[idx].set_title(title_text)
    axes[idx].axis('off')
    
    out = pred_dir / f"prediction_{r.image_id_str}.jpg"
    im.save(out)
    rows.append({'image_id': r.image_id_str, 'true_class': r['class'], 'predicted_class': classes[pred_idx], 'confidence': float(pr[pred_idx]), 'true_box': true, 'pred_box': pred})

plt.tight_layout()
plt.show()
pd.DataFrame(rows).to_csv(OUT / 'sample_predictions.csv', index=False)


## 5. Tesla Deaths EDA (Exploratory Data Analysis)

In [ ]:
print('Running Tesla EDA...', flush=True)
raw = pd.read_csv(deaths_src)
raw.columns = [c.strip() for c in raw.columns]
df = raw.dropna(how='all').copy()

if 'Case #' in df:
    df = df[df['Case #'].notna()].copy()
if 'Date' in df:
    df['Date_parsed'] = pd.to_datetime(df['Date'], errors='coerce')
for c in df.columns:
    if c not in ['Date', 'Date_parsed', 'Country', 'State', 'Description', 'Source', 'Note', 'Deceased 1', 'Deceased 2', 'Deceased 3', 'Deceased 4', 'Model']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df.to_csv(OUT / 'tesla_deaths_cleaned.csv', index=False)

eda = {
    'raw_rows': int(raw.shape[0]),
    'usable_case_rows': int(df.shape[0]),
    'total_reported_deaths': float(df.get('Deaths', pd.Series(dtype=float)).fillna(0).sum()),
    'tesla_driver_death_events': int((df.get('Tesla driver', pd.Series(dtype=float)).fillna(0) > 0).sum()),
    'tesla_occupant_death_events': int((df.get('Tesla Occupant', pd.Series(dtype=float)).fillna(0) > 0).sum()),
    'cyclist_or_pedestrian_events': int((df.get('Cyclists/ Peds', pd.Series(dtype=float)).fillna(0) > 0).sum()),
    'other_vehicle_collision_events': int((df.get('Other vehicle', pd.Series(dtype=float)).fillna(0) > 0).sum()),
    'verified_autopilot_death_sum': float(df.get('Verified Tesla Autopilot Deaths', pd.Series(dtype=float)).fillna(0).sum())
}
json.dump(eda, open(OUT / 'tesla_eda_summary.json', 'w'), indent=2)
print("Tesla Safety EDA Stats:")
print(json.dumps(eda, indent=2))


### Plot Fatalities Distribution Curves

In [ ]:
plots = [
    ('Year', 'tesla_events_by_year', 'Tesla Death Events by Year', None),
    ('Country', 'tesla_events_by_country', 'Top Countries by Event Count', 10),
    ('State', 'tesla_events_by_state', 'Top States by Event Count', 15),
    ('Deaths', 'deaths_per_event_distribution', 'Deaths per Accident Event', None),
    ('Model', 'tesla_events_by_model', 'Event Distribution by Tesla Model', 10),
    ('Verified Tesla Autopilot Deaths', 'verified_autopilot_deaths_distribution', 'Verified Tesla Autopilot Deaths per Event', None)
]

for col, fname, title, top in plots:
    if col in df:
        vals = df[col].replace('-', np.nan).dropna()
        if col in ['Year', 'Deaths', 'Verified Tesla Autopilot Deaths']:
            vals = vals.astype(int).value_counts().sort_index()
        else:
            vals = vals.astype(str).value_counts().head(top)
        if len(vals) > 0:
            plt.figure(figsize=(10, 4))
            vals.plot(kind='bar', color='darkcyan', title=title)
            plt.ylabel('Count')
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.savefig(FIG / f'{fname}.png', dpi=150)
            plt.show()


## 6. Project Packaging and Export

In [ ]:
readme = f"""# RoadVision AI: Vehicle Detection and Autopilot Safety Analytics\n\n## Executed local end-to-end result\nThis repository contains the local execution baseline.\n\n## Model test metrics\n- Classification accuracy: {metrics['test']['accuracy']:.3f}\n- Mean IoU: {metrics['test']['mean_iou']:.3f}\n- Combined loss: {metrics['test']['loss']:.3f}\n"""
(BASE / 'README.md').write_text(readme)
(BASE / 'requirements.txt').write_text('pandas\nnumpy\nmatplotlib\nscikit-learn\npillow\ntorch\ntorchvision\n')

zip_path = BASE.parent / 'roadvision_ai_executed_project.zip'
print(f"Creating project package at {zip_path}...", flush=True)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in BASE.rglob('*'):
        if path.is_file():
            rel = path.relative_to(BASE)
            if str(rel).startswith('data/Images/') or rel.name in ['labels.csv', 'Tesla - Deaths.csv', '1739779188_capstone1problemstatement.pdf', 'roadvision_ai_executed_project.zip']:
                continue
            z.write(path, Path('roadvision_ai_executed') / rel)
print('DONE', zip_path, flush=True)
